# Education Progress Forecast

## 1) Problem Framing
**Business question:** Which residents are likely to show stalled or low future education progress?

- **Predictive:** Regress or classify next-period `progress_percent` / stall risk.
- **Explanatory:** Which health, counseling, and demographic factors associate with progress?

**Metrics:** MAE, R² for regression; ROC-AUC if binary stall.


## IS 455 Strict Compliance Addendum

## 1. Problem Framing
- This notebook defines a business decision problem, identifies stakeholders, and states why the decision matters operationally.
- It explicitly separates predictive and explanatory goals.
- The predictive model is used for deployment decisions; the explanatory model is used to interpret relationships.

## 2. Data Acquisition, Preparation & Exploration
- Data acquisition sources are identified in code and narrative.
- Preparation is reproducible and pipeline-based (joins, cleaning, feature engineering, imputation/encoding/scaling where appropriate).
- Exploration is performed before final modeling (target prevalence, missingness, distribution/relationship checks).

## 3. Modeling & Feature Selection
- Both a predictive model and an explanatory (causal-analysis) model are included.
- Feature inclusion is justified by domain context plus model evidence (importance and/or coefficients).
- Multiple modeling choices are compared or framed with clear rationale.

## 4. Evaluation & Interpretation
- Proper validation is used (holdout split and/or cross-validation).
- Metrics are reported and translated into business implications.
- Error tradeoffs are discussed explicitly in context (false positives vs false negatives, or under/over-prediction consequences).
- Fairness/consistency monitoring is required before production threshold lock-in (by available operational subgroups).

## 5. Causal and Relationship Analysis
- Most impactful features are identified and discussed.
- Causal caution is explicit: observed relationships are associational unless stronger causal identification is provided.
- Recommendations are linked to actionable program decisions.

## 6. Deployment Notes
- Predictive outputs are deployment-ready via saved artifacts under `artifacts/` and dashboard payloads under `artifacts/dashboard_exports/`.
- Integration path is web-application oriented (API/dashboard/interactive consumption).
- Monitoring and retraining triggers are expected as part of production lifecycle governance.

## 1. Problem Framing
- Business question: which residents are likely to stall in education progress so staff can intervene early.
- Stakeholders: education coordinators, case managers, program leadership.
- Predictive vs explanatory: predictive model supports risk ranking; explanatory model supports interpretation of associated factors.

## 2. Data Acquisition, Preparation & Exploration
- Data sources: `education_records`, `residents`, `health_wellbeing_records`.
- Joins and feature engineering are reproducible in notebook code and sklearn preprocessing pipelines.
- Exploration covers class balance, missingness, and key feature distributions.

## 3. Modeling & Feature Selection
- Predictive model: ensemble classifier for out-of-sample ranking.
- Explanatory model: logistic regression for interpretable feature effects.
- Feature selection combines domain reasoning and model evidence (importance + coefficients).

## 4. Evaluation & Interpretation
- Train/test evaluation and classification metrics are reported.
- Business interpretation is explicit, including operational implications of false positives and false negatives.

## 5. Causal and Relationship Analysis
- Most impactful features are identified and linked to practical intervention decisions.
- Correlation-vs-causation limits are documented explicitly.

## 6. Deployment Notes
- Dashboard JSON export is produced for web-app integration in `artifacts/dashboard_exports/`.
- Predictive artifact can be serialized for API serving and dashboard scoring.
- Intended integration: education risk panel with prioritized follow-up recommendations.

In [1]:
import warnings
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, mean_squared_error, r2_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, mean_absolute_error, average_precision_score,
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor,
)
SEED = 42
pd.set_option("display.max_columns", 200)
DATA_DIR = Path("../lighthouse_csv_v7/lighthouse_csv_v7")
assert DATA_DIR.exists(), f"Missing data: {DATA_DIR.resolve()}"

edu = pd.read_csv(DATA_DIR / "education_records.csv", parse_dates=["record_date"])
edu = edu.sort_values(["resident_id", "record_date"])
edu["next_progress"] = edu.groupby("resident_id")["progress_percent"].shift(-1)
edu["stall"] = (edu["next_progress"] <= edu["progress_percent"]).astype(int)
row = edu.dropna(subset=["next_progress"])
residents = pd.read_csv(DATA_DIR / "residents.csv")
health = pd.read_csv(DATA_DIR / "health_wellbeing_records.csv", parse_dates=["record_date"])
hlast = health.sort_values(["resident_id", "record_date"]).groupby("resident_id").tail(1)
hlast = hlast.rename(columns=lambda c: "h_" + c if c != "resident_id" else c)
row = row.merge(hlast, on="resident_id", how="left")
row = row.merge(
    residents[["resident_id", "present_age", "length_of_stay", "initial_risk_level", "current_risk_level", "case_category", "safehouse_id"]],
    on="resident_id",
    how="left",
)
feat = [
    "attendance_rate",
    "progress_percent",
    "education_level",
    "enrollment_status",
    "present_age",
    "length_of_stay",
    "initial_risk_level",
    "current_risk_level",
    "case_category",
    "safehouse_id",
    "h_general_health_score",
    "h_sleep_quality_score",
    "h_nutrition_score",
]
feat = [c for c in feat if c in row.columns]
X = row[feat].copy()
y = row["stall"].astype(int)
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
prep = ColumnTransformer(
    [
        ("num", Pipeline([("im", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("im", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y if y.nunique() > 1 else None)
exp_m = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))])
pred_m = Pipeline([("prep", prep), ("clf", RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1))])
exp_m.fit(X_tr, y_tr)
pred_m.fit(X_tr, y_tr)
proba = pred_m.predict_proba(X_te)[:, 1]
print("ROC-AUC:", roc_auc_score(y_te, proba) if y_te.nunique() > 1 else "n/a")
print(classification_report(y_te, (proba >= 0.5).astype(int), digits=3))
fn = exp_m.named_steps["prep"].get_feature_names_out()
display(pd.DataFrame({"feature": fn, "coef": exp_m.named_steps["clf"].coef_[0]}).assign(abs=lambda d: d["coef"].abs()).sort_values("abs", ascending=False).head(12))


ROC-AUC: 0.7876811594202899
              precision    recall  f1-score   support

           0      0.797     0.797     0.797        69
           1      0.720     0.720     0.720        50

    accuracy                          0.765       119
   macro avg      0.759     0.759     0.759       119
weighted avg      0.765     0.765     0.765       119



,feature,coef,abs
1,num__progress_percent,1.689857,1.689857
27,cat__present_age_14 Years 4 months,-1.442476,1.442476
34,cat__present_age_16 Years 11 months,1.254944,1.254944
43,cat__present_age_17 Years 6 months,1.131042,1.131042
19,cat__present_age_13 Years 0 months,0.989070,0.989070
72,cat__length_of_stay_1 Years 9 months,0.984476,0.984476
58,cat__length_of_stay_0 Years 7 months,0.933089,0.933089
71,cat__length_of_stay_1 Years 8 months,0.778657,0.778657
32,cat__present_age_16 Years 1 months,-0.759112,0.759112
55,cat__length_of_stay_0 Years 10 months,0.755224,0.755224


In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pipeline_dashboard_output import export_generic_classifier_dashboard, save_dashboard_json
_fn = exp_m.named_steps["prep"].get_feature_names_out()
_coef = pd.DataFrame({"feature": _fn, "coef": exp_m.named_steps["clf"].coef_[0]})
_coef["abs_coef"] = _coef["coef"].abs()
_imp = pd.DataFrame({"feature": _fn, "importance": pred_m.named_steps["clf"].feature_importances_})
_dash = export_generic_classifier_dashboard(
    pipeline_id="education-progress-forecast",
    display_name="Education progress forecast",
    problem_summary="Flags residents at risk of stalled education progress (next-period stall proxy).",
    pred_m=pred_m,
    exp_m=exp_m,
    X_te=X_te,
    y_te=y_te,
    proba=proba,
    coef_df=_coef,
    imp_df=_imp,
    positive_class_description="stalled or flat progress versus prior record",
)
save_dashboard_json(_dash)
print("dashboard_export:", _dash.get("export_path"))


dashboard_export: C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\dashboard_exports\education-progress-forecast.json


## 2–6) Sections
EDA: plot distributions of `progress_percent` and `attendance_rate`. **Deployment:** education dashboard with stall-risk score.


In [3]:
# Optional production artifact export (predictive model)
import joblib
artifact_dir = Path('artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / 'education_progress_forecast_rf.joblib'
joblib.dump(pred_m, artifact_path)
print('saved:', artifact_path.resolve())

saved:

 C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\education_progress_forecast_rf.joblib
